In [2]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [31]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("C:/Mate academy/py-restaurant-data-analysis/db.sqlite3")

orders = pd.read_sql("SELECT * FROM restaurant_order", conn)
order_items = pd.read_sql("SELECT * FROM restaurant_orderitem", conn)
products = pd.read_sql("SELECT * FROM restaurant_product", conn)

print("Orders:", orders.shape)
print("OrderItems:", order_items.shape)
print("Products:", products.shape)


Orders: (13397, 2)
OrderItems: (74818, 4)
Products: (248, 3)


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [32]:
merged = order_items.merge(products, left_on="product_id", right_on="id")

top10_quantity = (
    merged.groupby("name")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top10_df = top10_quantity.reset_index()
top10_df["percentage"] = (top10_df["quantity"] / top10_df["quantity"].sum()) * 100
print(top10_df)

                   name  quantity  percentage
0         Plain Papadum     10648   26.766547
1            Pilau Rice      6367   16.005128
2            Plain Naan      4983   12.526080
3           Garlic Naan      3318    8.340665
4            Plain Rice      2964    7.450793
5          Onion Bhajee      2749    6.910334
6         Mango Chutney      2504    6.294462
7  Chicken Tikka Masala      2473    6.216536
8               Chapati      1935    4.864131
9            Mint Sauce      1840    4.625324


# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [33]:
merged["item_price"] = merged["price"] * merged["quantity"]

top10_revenue = (
    merged.groupby("name")["item_price"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top10_revenue_df = top10_revenue.reset_index()
top10_revenue_df["percentage"] = (top10_revenue_df["item_price"] / top10_revenue_df["item_price"].sum()) * 100
print(top10_revenue_df)

                   name  item_price  percentage
0  Chicken Tikka Masala    22133.35   17.454050
1            Pilau Rice    18782.65   14.811735
2            Plain Naan    12955.80   10.216763
3                 Korma    12261.50    9.669247
4           Bombay Aloo    10894.45    8.591211
5          Onion Bhajee    10858.55    8.562901
6        Butter Chicken    10626.60    8.379988
7           Garlic Naan     9788.10    7.718759
8       Korma - Chicken     9764.45    7.700109
9            Plain Rice     8743.80    6.895238


# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [41]:
merged['item_price'] = merged['price'] * merged['quantity']

merged['datetime'] = pd.to_datetime(merged['datetime'])

merged['order_hour'] = merged['datetime'].dt.hour

income_by_hour = merged.groupby('order_hour')['item_price'].sum().reset_index()

print(income_by_hour)

    order_hour  item_price
0            0      177.95
1            1       54.65
2            2      199.25
3            3        8.90
4            4       63.45
5            5       57.40
6            6      121.60
7            8      447.70
8            9      570.60
9           10     1250.35
10          11     3807.05
11          12    10565.85
12          13     8282.65
13          14     3343.55
14          15     3781.70
15          16    15634.75
16          17    72110.20
17          18   132462.50
18          19   109045.05
19          20    50218.25
20          21    21480.30
21          22    11001.50
22          23      373.25


# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [42]:
merged['datetime'] = pd.to_datetime(merged['datetime'])

merged['order_day_of_week'] = merged['datetime'].dt.dayofweek

income_by_day = merged.groupby('order_day_of_week')['item_price'].sum().reset_index()

day_map = {0: 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday',
           4: 'Friday', 5: 'Saturday', 6: 'Sunday'}
income_by_day['order_day_of_week'] = income_by_day['order_day_of_week'].map(day_map)

print(income_by_day)

  order_day_of_week  item_price
0            Monday    40008.30
1           Tuesday    38145.65
2         Wednesday    41246.20
3          Thursday    46021.55
4            Friday   100339.15
5          Saturday   112191.65
6            Sunday    67105.95
